# Basic Data Quality Checks for a Reserving Data Pipeline
This notebook goes through some data quality checks that we should consider in a reserving pipeline to ensure our data meets expectations as per the data quality dimensions, including:
- testing that data types and ranges are appropriate (validity)
- checking for completeness and missing data
- confirming uniqueness where required
- ensuring consistency within the dataset
- verifying timeliness of data (nothing extraneous and nothing missing due to delays)

Note: The data used is for demonstration purposes only and should not be relied upon for other analyses.

In [1]:
# import packages:

from platform import python_version
import numpy as np
import pandas as pd
import duckdb
import json # built-in library
from datetime import datetime, timedelta # built-in libraries

# verify that package versions are as expected:

print('Python version:', python_version())
print('Numpy version:', np.__version__)
print('Pandas version:', np.__version__)
print('Duckdb version:', duckdb.__version__)

pd.set_option('display.max_columns', None) # don't hide columns in display

Python version: 3.13.12
Numpy version: 2.4.2
Pandas version: 2.4.2
Duckdb version: 1.4.3


In [ ]:
# specify input files here:

CONFIG_FILE_PATH = 'config_comp_2020.json'
EXPECTED_VERSION = 1 # this version of the data pipeline requires the config file to be version 1
CLAIMS_FILE_PATH = 'Datasets/claim_details.parquet'
POLICY_FILE_PATH = 'Datasets/policy_details.parquet'



In [3]:
# load in config file first and check that the version matches what is expected. These will be used as hardcoded values in the remainder of the pipeline.
# for future runs, we'll be able to edit just the config file without having to touch the code
with open(CONFIG_FILE_PATH, 'r') as f:
    settings = json.load(f)

print(settings)

assert settings['version'] == EXPECTED_VERSION, 'Version of config file is invalid. Expected: {EXPECTED_VERSION}. Actual: {settings["version"]}'

{'version': 1, 'valuation_date': '2021-01-01', 'earliest_accident_year': 2015, 'latest_accident_year': 2020, 'subline': 'COMP'}


In [4]:
# let's first get the claims table and filter it appropriately:
# (Note: we will pretend that date of intimation represents valuation date and intimation amount is paid loss)

claims = duckdb.sql(f'''
    SELECT
        *
    FROM
        read_parquet('{CLAIMS_FILE_PATH}')
    WHERE
        policytype = '{settings["subline"]}'
        AND YEAR(date_of_accident) >= {settings["earliest_accident_year"]}
        AND YEAR(date_of_accident) <= {settings["latest_accident_year"]}
        AND date_of_intimation < DATE '{settings["valuation_date"]}'
    ;
''').df()


In [5]:
# let's run some basic validation checks:

# firstly, we may require that claim number is not null and unique in the data:
if claims[['CLAIM_NO']].drop_duplicates().shape[0] < claims.shape[0]:
    raise ValueError('CLAIM_NO is not unique in the claims dataset!')

ValueError: CLAIM_NO is not unique in the claims dataset!

In [6]:
if claims['CLAIM_NO'].isna().sum() > 0:
    raise ValueError(f"CLAIM_NO has {claims['CLAIM_NO'].isna().sum()} null values in the claims dataset!")

ValueError: CLAIM_NO has 34 null values in the claims dataset!

In [8]:
# the tests above failed! In practice, we should look into why some records would have unexpected
# duplicate or empty information for now, we'll exclude the handful of problematic records.
duplicated = claims['CLAIM_NO'].value_counts()
claims = claims[(claims['CLAIM_NO'].notna()) & (~claims['CLAIM_NO'].isin(duplicated[duplicated > 1].index))]


In [9]:
# suppose previous runs of the data included many regions. If we notice that certain regions like 'DUBAI'
# typically make up around X% of claims, we can add a subjective, but reasonable ranged check to test
# that we're not excluding regions. This can happen for several reasons. If different regions are sourced
# from a different claims reporting system and one pull failed, we'll be missing essential data (impairing
# completeness). It's also possible that the code for regions changed at some point (e.g. AD -> Abu Dhabi).
# These tests can take some time to specify initially, so the best fields for this type of test would be
# the ones directly used in a reserve analysis. Note that this won't be perfect - we can still occassionally
# get false alarms. We suggest choosing wide intervals.

by_region = claims.groupby('REG')['CLAIM_NO'].count()
by_region /= by_region.sum() # get percentages
if not 0.6 < by_region['DUBAI'] < 0.9:
    raise ValueError("Percentage of claims occurring in 'DUBAI' is outside a reasonable range. Actual {by_region['DUBAI']} is not within 0.6 and 0.9")
if not 0.05 < by_region['SHJ'] < 0.15:
    raise ValueError("Percentage of claims occurring in 'SHJ' is outside a reasonable range. Actual {by_region['DUBAI']} is not within 0.05 and 0.15")
if not 0.01 < by_region['AD'] < 0.1:
    raise ValueError("Percentage of claims occurring in 'SHJ' is outside a reasonable range. Actual {by_region['DUBAI']} is not within 0.01 and 0.1")


In [11]:
# if we group by accident year and month, we should expect to see values in each accident period and in each valuation period:
# the most common issue we've seen here is that the earliest or latest loss information is sometimes excluded
# fortunately, these tests will all pass without incident

claims['accident_yearmo'] = claims['DATE_OF_ACCIDENT'].dt.year.astype(str) + '-' + claims['DATE_OF_ACCIDENT'].dt.month.astype(str)
claims['valuation_yearmo'] = claims['DATE_OF_INTIMATION'].dt.year.astype(str) + '-' + claims['DATE_OF_INTIMATION'].dt.month.astype(str)

latest_acc_month = str(settings['latest_accident_year']) + '-12' # last month of the year
latest_val_month = (datetime.strptime(settings['valuation_date'], '%Y-%m-%d') - timedelta(days=1)).strftime('%Y-%m') # get the previous month

if latest_acc_month not in list(claims['accident_yearmo'].unique()):
    raise ValueError(f'The latest accident month {latest_acc_month} is not in the data!')

if latest_val_month not in list(claims['valuation_yearmo'].unique()):
    raise ValueError(f'The latest valuation month {latest_val_month} is not in the data!')

if not (claims.groupby('accident_yearmo').agg({'INTIMATED_AMOUNT': 'sum'}) > 0).all().all():
    raise ValueError(f'There are accident months for which there are no losses!')

if not (claims.groupby('valuation_yearmo').agg({'INTIMATED_AMOUNT': 'sum'}) > 0).all().all():
    raise ValueError(f'There are valuation months for which there are no losses!')

In [12]:
# we can also consider the consistency of columns relative to each other.
# It's reasonable to say intimation date for a claim should be no earlier than accident date:

if (claims['DATE_OF_ACCIDENT'] > claims['DATE_OF_INTIMATION']).any():
    raise ValueError(f"There are {(claims['DATE_OF_ACCIDENT'] > claims['DATE_OF_INTIMATION']).sum()} claims recorded before their accident date!")

ValueError: There are 7 claims recorded before their accident date!

In [13]:
# we'll throw out those claims, but again, this type of issue requires a conversation with upstream data providers
claims = claims[claims['DATE_OF_ACCIDENT'] <= claims['DATE_OF_INTIMATION']]

In [14]:
# for the purposes of this example, let's confirm that the paid loss is within a reasonable range,
# which we shall define as always non-negative and capped at 1M:

if (claims['INTIMATED_AMOUNT'] < 0).any() or (claims['INTIMATED_AMOUNT'] > 1_000_000).any():
    raise ValueError('Some paid amounts are negative!')

# if desired, we could also check that numbers aren't always rounded, if we don't expect them to be (if we expect loss amounts
# to include cents, but they always show up as rounded to the nearest integer, there could be a small problem upstream)
if ((claims['INTIMATED_AMOUNT'].isna()) | (np.round(claims['INTIMATED_AMOUNT'], 0) == np.round(claims['INTIMATED_AMOUNT'], 2))).all():
    raise ValueError('All paid amounts seem to be rounded to the dollar, which is not expected!')

# there are many more checks we could make, depending on how thorough we want to be, but we'll stop here.
# Generally, our advice is that if you notice an issue in one run, aim to add a check would identify similar issues in the future.

In [15]:
claims.to_parquet('FinalDatasets/claims_filtered_step1.parquet', index=False) # save the intermediate table for use later

In [16]:
# now join the claims table onto the policy dataset to get premium information
# we'll check row counts before and after the left join to confirm it worked

claims_with_premium = duckdb.sql(f'''
    SELECT
        claims.*,
        policy.premium2
    FROM
        read_parquet('FinalDatasets/claims_filtered_step1.parquet') AS claims
    LEFT JOIN
        read_parquet('{POLICY_FILE_PATH}') AS policy
    ON
        claims.policy_no = policy.policy_no
        AND claims.policy_start = policy.pol_eff_date
    ;
''').df()

before_join_row_count = claims.shape[0]
after_join_row_count = claims_with_premium.shape[0]

# the following line will trigger and stop subsequent execution because there are duplicates!
if after_join_row_count > before_join_row_count:
    raise ValueError(f'Joining claims and policy details created duplicates! Expected: {before_join_row_count}. Actual: {after_join_row_count}')

display(claims_with_premium.head())

ValueError: Joining claims and policy details created duplicates! Expected: 45642. Actual: 45644

In [17]:
# we'll fix the issue for now by excluding the extraneous records. In practice, this requires a conversation with
# the team that provides the policy details table to investigate why duplicates exist and which record is correct

claims_with_premium = claims_with_premium.drop_duplicates(['POLICY_NO', 'POLICY_START'])
# the check from above won't trigger this time
if claims_with_premium.shape[0] > before_join_row_count:
    raise ValueError(f'Joining claims and policy details created duplicates! Expected: {before_join_row_count}. Actual: {claims_with_premium.shape[0]}')

display(claims_with_premium)

,Account_Code,DATE_OF_INTIMATION,DATE_OF_ACCIDENT,PLACE_OF_LOSS,CLAIM_NO,AGE,TYPE,DRIVING_LICENSE_ISSUE,BODY_TYPE,MAKE,MODEL,YEAR,CHASIS_NO,REG,SUM_INSURED,POLICY_NO,POLICY_START,POLICY_END,INTIMATED_AMOUNT,INTIMATED_SF,EXECUTIVE,PRODUCT,POLICYTYPE,NATIONALITY,accident_yearmo,valuation_yearmo,PREMIUM2
0,5284,2015-01-21 11:49:00,2015-01-18,SHARJAH,DU/10/PC/COMP/1924/15,40.0,RECOVERY CLAIM,01-01-1900,4 WD,SUZUKI,GRAND VITARA,2012.0,JS3TD04V4C4103329,AJMAN,60000.0,102049047,2015-01-16,2016-02-15,0.0,0.0,SUVARNA,NOT CLASSIFIED,COMP,NaN,2015-1,2015-1,1910.0
1,6656,2015-01-26 16:06:00,2015-01-24,SHARJAH,DU/10/PC/COMP/1974/15,32.0,RECOVERY CLAIM,01-01-1900,4 WD,TOYOTA,LAND CRUISER,2013.0,JTMHU09J5D5067706,DUBAI,140000.0,102049037,2015-01-07,2016-02-06,9600.0,300.0,BR,TOURISM,COMP,NaN,2015-1,2015-1,4530.0
2,7492,2015-01-28 16:39:00,2015-01-23,DUBAI,DU/10/PC/COMP/1990/15,32.0,RECOVERY CLAIM,01-01-1900,SALOON,NISSAN,TIIDA,2011.0,3N1BC1C62BK214752,DUBAI,26000.0,102049169,2015-01-13,2016-02-12,650.0,0.0,RAHEEM,STANDARD,COMP,NaN,2015-1,2015-1,1090.0
3,4095,2015-02-03 09:46:00,2015-02-01,DUBAI,DU/10/PC/COMP/2002/15,44.0,RECOVERY CLAIM,01-01-1900,AS PER FLEET LIST,VEHICLE AS PER,SPREAD SHEET,NaN,LIST ATTACHED,RAK,2621786.0,102049444,2015-01-25,2016-01-24,7000.0,300.0,AMIT,NOT CLASSIFIED,COMP,NaN,2015-2,2015-2,85845.0
4,4259,2015-02-09 09:48:00,2015-02-05,DUBAI,DU/10/PC/COMP/2037/15,31.0,OD Claim,01-01-1900,SALOON,MERCEDES,S350,2003.0,WDB2201671A343814,DUBAI,33000.0,102049111,2015-01-12,2016-01-11,350.0,0.0,MANOHAR,NOT CLASSIFIED,COMP,NaN,2015-2,2015-2,1400.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45635,903,2019-01-27 16:38:00,2018-11-08,DUBAI,DU/10/PC/COMP/5842/19,0.0,TP Claim,01-01-1900,LUXURY PICKUP,FORD,F-150,2012.0,1FTFW1EF1CFA00053,DUBAI,50000.0,102122931,2018-09-24,2019-10-23,2200.0,NaN,BR,STANDARD,COMP,EGYPTION,2018-11,2019-1,NaN
45640,7957,2019-09-16 12:12:00,2019-09-12,ABU DHABI,DU/10/PC/COMP/2472/20,31.0,OD Claim,01-01-1900,COUPE,NISSAN,INFINITY Q60,2018.0,JN1CV7EK6JM800039,AD,122240.0,102126619,2019-02-04,2020-03-03,40000.0,500.0,BR,M 2.5,COMP,PALESTINE,2019-9,2019-9,NaN
45641,7957,2019-11-24 09:30:00,2019-11-13,RAS AL KHAIMAH,DU/10/PC/COMP/3605/20,0.0,TP Claim Ins,01-01-1900,SALOON,NISSAN,ALTIMA,2012.0,1N4AL2A9XCC207704,RAK,15173.0,102136656,2019-11-16,2020-12-15,8600.0,NaN,BR,M 2019,COMP,INDIAN,2019-11,2019-11,NaN
45642,9558,2020-07-29 12:06:00,2020-07-28,DUBAI,DU/10/PC/COMP/1001/21,28.0,OD Claim,25-03-2010,SALOON,TESLA,3,2020.0,5YJ3E7EC4LF683031,DUBAI,243780.0,102142954,2020-06-10,2021-07-09,5000.0,150.0,BR,M 2019,COMP,EMIRATE,2020-7,2020-7,NaN


In [18]:
# we'll check that each claim has a matching premium associated with it:

# this check will fail, but we won't need for the development triangles we create below, so we'll ignore it
if claims_with_premium['PREMIUM2'].isna().any():
    raise ValueError(f"There are {claims_with_premium['PREMIUM2'].isna().sum()} claims with a missing premiums!")

ValueError: There are 1048 claims with a missing premiums!

In [19]:
# now we'll put together incremental loss and count triangles:

claims_with_premium['ay'] = claims_with_premium['DATE_OF_ACCIDENT'].dt.year
claims_with_premium['aq'] = claims_with_premium['DATE_OF_ACCIDENT'].dt.quarter
claims_with_premium['vy'] = claims_with_premium['DATE_OF_INTIMATION'].dt.year
claims_with_premium['vq'] = claims_with_premium['DATE_OF_INTIMATION'].dt.quarter
claims_with_premium['dev'] = (claims_with_premium['vy'] - claims_with_premium['ay']) * 12 + 12
claims_with_premium['rept_count'] = 1
triangle_flat = claims_with_premium.groupby(['ay', 'dev']).agg({'INTIMATED_AMOUNT': 'sum', 'rept_count': 'sum'}).reset_index()
paid_triangle = triangle_flat.pivot(index='ay', columns='dev', values='INTIMATED_AMOUNT').fillna(0)
rept_cnt_triangle = triangle_flat.pivot(index='ay', columns='dev', values='rept_count').fillna(0)
display(paid_triangle)
display(rept_cnt_triangle)

dev,12,24,36,48,60
ay,,,,,
2015,2.363146e+07,1810520.0,53762.0,19911.0,2550.0
2016,5.184399e+07,2651532.0,137548.0,32884.0,5200.0
2017,3.952643e+07,1467797.0,50265.0,14154.0,0.0
2018,2.444508e+07,1746431.0,41341.0,0.0,0.0
2019,1.689493e+07,524989.0,0.0,0.0,0.0
2020,1.554393e+07,0.0,0.0,0.0,0.0


dev,12,24,36,48,60
ay,,,,,
2015,3499.0,527.0,27.0,8.0,2.0
2016,8181.0,749.0,68.0,11.0,2.0
2017,5961.0,475.0,24.0,5.0,0.0
2018,3298.0,320.0,13.0,0.0,0.0
2019,2859.0,184.0,0.0,0.0,0.0
2020,2490.0,0.0,0.0,0.0,0.0
